In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from eucare.base import *
import eucare.plotting as euplot


plt.figure(figsize=(5, 5))
prev_poly = regular_poly_points(3)
for i in range(3, 20):
    poly = regular_poly_points(i) * np.random.rand()
    j = np.random.randint(0, i-2)
    mat = find_affine(poly[j:j+2][::-1], prev_poly[i-3:i-1])
    poly = apply_affine(poly, mat)
    euplot.plot_polygon(poly)
    prev_poly=poly
    #plt.scatter(*(poly[j:j+2].T))
    
euplot.set_equal_aspect()

## HalfEdge Data Structure

In [ ]:
from eucare.half import AttributeObject

a = AttributeObject()

print(a.has_attributes())
a['adsf'] = 'party'
print(a.has_attributes())
for key, val in a.items():
    print(key, val)


In [ ]:
from eucare.half import EuclideanPositionHEG, HalfEdge, Face, rotate_by
from eucare.base import angle_to_axis
from eucare.redering import CairoRenderer
import numpy as np
from copy import copy
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


    
#for v in G.nodes:
    #print(v, G[v])
    




In [ ]:
from eucare.half import Vertex, CyclicHalfedgeGraph, IdObject
from tqdm import tqdm_notebook as tqdm
import networkx as nx

IdObject.reset_ids()
poly = CyclicHalfedgeGraph([Vertex() for i in range(4)])
#for v in poly.vertices:
#    print(v)
#    for h in v.outgoing_iter():
#        pass
#        print(h.orig, h.dest)

#hs = list(poly.halfedges)

#f = any_element(poly.faces)
#print(f.__dict__)
#[print(v) for v in f.reverse_halfedge_iter()]


for i in range(2):
    print(i, poly.order)
    border_vertices = poly.border_vertices()
    for h1 in tqdm(list(poly.border_edge_iter())):
        #print(h1)
        to_attach = CyclicHalfedgeGraph([Vertex() for i in range(5)])
        h2 = to_attach.get_any_border()
        poly.add_graph(to_attach)
        poly.glue_e2e(h1, h2)
    for v in border_vertices:
        poly.close_vertex(v)

G = poly.to_networkx_undirected()
pos = nx.spring_layout(G.to_undirected())
nx.draw_networkx_nodes(G, pos, cmap=plt.get_cmap('jet'), node_size=500)
nx.draw_networkx_edges(G, pos, edge_color='r', arrows=True)
nx.draw_networkx_labels(G, pos)
plt.show()

heg = EHEG_from_layout(G, pos)
heg.check_consistency()

for v in G.nodes:
    v['pos'] = pos[v]
    
rend = CairoRenderer(width=1500, scale=500, face_inset=0.02, line_width=0.02)
surface = rend.render_graph(heg)
filename = 'output.png'
surface.write_to_png(filename)
surface.finish()
img = mpimg.imread(filename)
plt.figure(figsize=(13, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:

    
    


G = gyro_graph()
for e in G.edges():
    print(e)
    print(G[e[0]][e[1]])

heg = EHEG_from_layout(G)
heg.check_consistency()

heg.show(scale=300, line_width=0.03, render_faces=False)

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
from eucare.half import HalfEdgeGraph, Vertex, IdObject, RegularNGon, CyclicHalfedgeGraph, InAngleHEG, EuclideanPositionHEG
from eucare.instructions import *
from eucare.base import unit_vector
from copy import deepcopy, copy
from tqdm import tqdm_notebook as tqdm
from eucare.plotting import plot_polygon
import matplotlib.pyplot as plt

    
#tile = copy(tile)

#def RegularNGon(n):
#    return CyclicHalfedgeGraph([Vertex() for _ in range(n)])

#n = 6
#proto_tile = RegularEuclideanTile(n, edge_labels = ['a'] * n)

def print_graph(graph):
    for e in graph.halfedges:
        print(e.__repr__(), e.on_border(), e.face)
        
def attatch_tile_instruction(proto_tile, label=None):
    def instruction(graph, edge):
        tile, edge_dict = proto_tile.make_graph()
        if label is not None:
            edge = edge_dict[label]
        else:
            # just take any edge
            edge = next(iter(edge_dict.values()))
        graph.glue_graph_e2e(tile, e, edge)
    return instruction

# define 6.4.3.4 tiling
hexagon = RegularEuclideanTile(6, edge_labels=['a', 'a', 'a', 'a', 'a', 'a'])
square = RegularEuclideanTile(4, edge_labels=['b', 'c', 'b', 'c'])
triangle = RegularEuclideanTile(3, edge_labels=['d', 'd', 'd'])
hexagon.edge_instructions['a'] = attatch_tile_instruction(square, 'b')
square.edge_instructions['b'] = attatch_tile_instruction(hexagon)
square.edge_instructions['c'] = attatch_tile_instruction(triangle)
triangle.edge_instructions['d'] = attatch_tile_instruction(square, 'c')
        

#tile = RegularNGon(n)
#print_graph(tile)
#for e in tile.border_edge_iter():
#    e['instruction'] = attatch_tile_instruction(proto_tile, 'a')
    

#for e in tile1.border_edge_iter():
#    e['instruction'] = instruction0
  

# possible problem: topology based merging can miss geometry

for k in tqdm(range(1)):
    IdObject.reset_ids()
    tiling = EuclideanPositionHEG(eps=1e-3, other=hexagon.make_graph(add_positions=True)[0])
    for i in range(1):
        for e in tiling.border_edges():
            if e.on_border() and e in tiling.halfedges:
                tiling.execute_edge_instruction(e)
                    
                #print('...',tiling.order,'...')

print('drawing layout')

for face in tiling.faces:
    points = np.stack([v['pos'] for v in face.vertex_iter()])
    plot_polygon(points)
plt.show()
#tiling.show_spring_layout()

In [ ]:
from eucare.base import unit_vector, angle_to_axis
unit_vector(np.pi/2)

In [ ]:
(8.37758 - 6.28318) / np.pi

In [ ]:
import numpy as np
import collections

class EuclideanVertex2D(Vertex):
    def __init__(self, pos, any_outgoing=None):
        super(EuclideanVertex2D, self).__init__(any_outgoing)
        if not isinstance(pos, colledctions.Sized):
            raise ValueError(f"position must be Sized. Got {pos}.")
        if len(pos) != 2:
            raise ValueError(f"Got position of length {len(pos)} != 2.")
        self.pos = np.array(pos, dtype=np.float32)
    
    @property
    def x(self):
        return self.pos[0]
    
    @property
    def y(self):
        return self.pos[1]

In [ ]:
v1, v2 = Vertex(), Vertex()
h1, h2 = HalfEdge(orig=v1, dest=v2), HalfEdge(orig=v2, dest=v1)
h1.rev = h2
h2.rev = h1
e = Edge(h1, h2)

In [ ]:
e[h1.orig]

In [ ]:
from copy import copy

v = Vertex()
d = dict()
d[v] = 3
v.any_outgoing = 123
d[v]

In [ ]:
from eucare.half import Vertex
from copy import deepcopy
a = Vertex()
a['a'] = a
b = deepcopy(a)
b['a']

In [ ]:
import cairo
width = 1000
height = 1000
surface = cairo.ImageSurface(
            cairo.FORMAT_RGB24, width, height)
dc = cairo.Context(surface)
dc.set_line_cap(cairo.LINE_CAP_ROUND)
dc.set_line_join(cairo.LINE_JOIN_ROUND)
dc.set_line_width(10)
dc.set_font_size(18.0)
dc.translate(width / 2, height / 2)
#dc.scale(self.scale, self.scale)
dc.set_source_rgb(0, 0, 0)
dc.paint()
surface.show_page()
surface.write_to_png('output.png')

from IPython.display import Image
Image(filename='output.png')

In [ ]:
{}

In [ ]:
from eucare.base import signed_area, to_barycentric_function

tri = np.array([[0, -1], [1, 0], [0, 1]])
conv = to_barycentric_function(tri)
conv([0, 0])

In [ ]:
%load_ext autoreload
%autoreload 2

from eucare.base import euclidean_to_barycentric_map
from eucare.conway import *
import numpy as np

g = gyro_graph((1/4, -1/4))
g.show()

tri = np.stack([g.v1['pos'], g.vf['pos'], g.v2['pos']])

def barycentric_to_euclidean_map(tri):
    def inner(barycentric_coords):
        return tri.T @ barycentric_coords
    return inner

e2b = euclidean_to_barycentric_map(tri)
b2e = barycentric_to_euclidean_map(tri)

for v in g.graph.vertices:
    print(v['pos'], e2b(v['pos']), b2e(e2b(v['pos'])))

In [ ]:
b2e(e2b([1, -2]))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from eucare.example_tilesets import t_3_3_4_3_4
from eucare.half import EuclideanPositionHEG, IdObject, HalfEdgeGraph
from eucare.conway import ambo_graph, truncate_graph, dual_graph, gyro_graph
from copy import deepcopy
from matplotlib import pyplot as plt
IdObject.reset_ids()

tiles = t_3_3_4_3_4()
G = EuclideanPositionHEG(eps=1e-6, other=tiles[-1].make_graph(add_positions=True)[0])
for _ in range(0):
    for h in G.border_edges():
        if h.on_border():
            G.execute_edge_instruction(h)
G.show(scale=100, line_width=0.1)

for op in [ambo_graph(), truncate_graph(), dual_graph(), gyro_graph()]:
    op.show()
    g = deepcopy(G)
    print(g.order), 
    g = op(g)
    print(g.order)
    g.show_spring_layout()
    #G.show(scale=300, line_width=0.02)
    g.check_consistency()

### Prototiles

In [ ]:
%load_ext autoreload
%autoreload 2
%pylab inline

In [ ]:
from eucare.half import *
from eucare.prototiles import *
render_settings = dict(line_width=0.06, face_inset=0.0,
                       render_edges=True, render_vertices=False, render_faces=True)

G, edgedict = RhombusTile().make_graph(add_positions=True)
G = EuclideanPositionHEG(other=G)

tile.attach_instruction(0)(G, edgedict[0])

G.show(**render_settings)

In [ ]:
from eucare.redering import CairoRenderer
renderer = CairoRenderer
renderer.autoscale(G)
ren

In [ ]:
class Classifier:
    """
    Class to classify stuff, e.g. faces by their number of sides, 
    or by some equivalence relation such as congruency of polygons
    """
    def __init__(self, save_items=True, save_indices=True):
        super(Classifier, self).__init__()
        
        self.used_indices = set()
        
        # option to keep track of a dict mapping classes to items
        self.save_items = save_items
        if self.save_items:
            self.items = dict()
        else:
            self.items = None
            
        # option to keep track of a dict mapping classes to items
        self.save_indices = save_indices
        if self.save_indices:
            self.saved_indices = dict()
        else:
            self.saved_indices = None

    def add_item(item):
        raise NotImplementedError
        
    def add_items(items):
        for item in items:
            self.add_item(item)
            
    def get_index(item):
        raise NotImplementedError
        
    def add_and_get_index(item):
        self.add_item(item)
        self.get_index(item)
        
        
class HashingClassifier(Classifier):
    """Computes a hash for each item. Items with the same hash belong to the same class."""
    
    def __init__(self, *super_args, **super_kwargs):
        super(HashingClassifier, self).__init__(*super_args, **super_kwargs)
        self.hash_to_index = dict()
    
    def compute_hash(item):
        return hash(item)
    
    def get_index(item):
        return self.compute_hash(item)
    

class RepresentationClassifier(Classifier):
    def __init__(self, *super_args, **super_kwargs):
        super(RepresentationClassifier, self).__init__(*super_args, **super_kwargs)
            
    def item_to_repr(item):
        return hash(item)
    
    def item_to_compare_repr(item):
        return self.to_repr(item)
    
    def compare_representations()
        

In [ ]:
import numpy as np
from eucare.classifiers import *

kwargs = dict(save_items=True, save_indices=True)
#c = CountingClassifier(SumClassifier(), save_items=True, save_indices=True)
c = CountingClassifier(NestedClassifier([lambda_classifier(lambda x: len(x)), SumClassifier, CyclicClassifier]), **kwargs)
#c = lambda_classifier(lambda x: len(x))(**kwargs)
print(c)
items = [(1, 0), (2, 0), (1, 0), (0, 1), (0, 1, 2), (1, 2, 0), (2, 1, 0), ((0, 2),)]
for item in items:
    print(f'{item}:\n {c.classify(item)}\n')
    
print(c.saved_items)
print(c.saved_indices)
if isinstance(c, RepresentationClassifier):
    print(f'reps: {c.count_to_repr}')

In [ ]:
import numpy as np
from eucare.classifiers import *
from eucare.example_graphs import *
from eucare.example_tilesets import *

class PreMapClassifier(Classifier):
    def __init__(self, other, func, *super_args, **super_kwargs):
        super(PreMapClassifier, self).__init__(*super_args, **super_kwargs)
        self.func = func
        self.other = other
        
    def _get_index(self, item):
        return self.other.classify(self.func(item))

    
class AdjacencyClassifier(CyclicClassifier):
    def __init__(self, key, *super_args, **super_kwargs):
        super(AdjacencyClassifier, self).__init__(tolerance=0, *super_args, **super_kwargs)
        self.key = key
        
    def _represent_query_item(self, item):
        return np.array([(f[self.key] if f is not None else None, item[self.key]) for f in item.face_iter()])

    
class VertexAdjacencyClassifier(CyclicClassifier):
    # TODO
    pass

def face_to_array(f):
    data = []
    for e in f.halfedge_iter():
        data.append((e['length'], e['in_angle']))
        #data.append(np.array(e.orig['pos'], dtype=np.float32))
    data = np.stack(data)
    #data -= np.mean(data, axis=0, keepdims=True)
    #print(data)
    return data


congruency_classifier = CountingClassifier(PreMapClassifier(
    NestedClassifier(
        [LenClassifier, SumClassifier, lambda: CyclicClassifier(allow_flip=False)]
    ),
    face_to_array))
adjacency_classifier = AdjacencyClassifier(key='color_key', allow_flip=False)

#G = rosette(50)
G = from_tiles(pgg_2x(), 13)
from eucare.conway import *

for f in G.faces:
    f['color_key'] = congruency_classifier.classify(f)
G.show(**plotting_kwargs)
    
G = dual_graph()(ambo_graph()((G)))
G.recompute_lengths_and_angles()

plotting_kwargs = dict(scale=100, line_width=0.01, face_inset=0, render_edges=True, render_vertices=False)
G.show(render_faces=False, **plotting_kwargs)

for f in G.faces:
    f['color_key'] = congruency_classifier.classify(f)

G.show(**plotting_kwargs)

for _ in range(1):
    for f in G.faces:
        f['pre_color_key'] = adjacency_classifier.classify(f)
    for f in G.faces:
        f['color_key'] = f['pre_color_key']
    
G.show(**plotting_kwargs)
    

In [ ]:
from eucare.example_graphs import rosette, pgg_2x_tiling
from copy import copy, deepcopy
G = pgg_2x_tiling(15)

set(key for n in G.vertices for key in list(n.attributes.keys()))

In [ ]:
%timeit pgg_2x_tiling(15)

In [ ]:
%timeit deepcopy(G)